##in this file, we want an alternate way of img cropping post h5-png conversion without title, and simpler file name.

#common sections

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#common cell!!

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import h5py   #part by part acess without loading
import numpy as np
import matplotlib.pyplot as plt
!pip install cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime
import gc
import re
from datetime import datetime


def extract_datetime_for_output(filename):
    """
    Extract date & time from filenames like:
    3RIMG_01JAN2023_0545_L2G_AOD_V02R00.h5

    Output format:
    02JAN2023_0715
    """
    match = re.search(r"(\d{2}[A-Z]{3}\d{4})_(\d{4})", filename)
    if match:
        date_part, time_part = match.groups()
        dt = datetime.strptime(
            f"{date_part}_{time_part}",
            "%d%b%Y_%H%M"
        )
        return dt.strftime("%d%b%Y_%H%M").upper()
    else:
        return "UNKNOWN_DATETIME"


# Mumbai bounding box
lat_min, lat_max = 18.8, 19.4
lon_min, lon_max = 72.7, 73.2

#no change

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 76.2 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

#no change

Mounted at /content/drive


In [ ]:
import os
import h5py   #part by part acess without loading
import numpy as np
import matplotlib.pyplot as plt
!pip install cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime
import gc
#no change

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 68.2 MB/s eta 0:00:00


In [ ]:
import re
from datetime import datetime

In [ ]:

def extract_datetime_for_output(filename):
    """
    Extract date & time from filenames like:
    3RIMG_01JAN2023_0545_L2G_AOD_V02R00.h5

    Output format:
    02JAN2023_0715
    """
    match = re.search(r"(\d{2}[A-Z]{3}\d{4})_(\d{4})", filename)
    if match:
        date_part, time_part = match.groups()
        dt = datetime.strptime(
            f"{date_part}_{time_part}",
            "%d%b%Y_%H%M"
        )
        return dt.strftime("%d%b%Y_%H%M").upper()
    else:
        return "UNKNOWN_DATETIME"


In [ ]:
# Mumbai bounding box
lat_min, lat_max = 18.8, 19.4
lon_min, lon_max = 72.7, 73.2

#no change

#23/01


In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 01_jan"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 01"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        # ❌ TITLE REMOVED (as requested)

        # ✅ Output filename: DATE-TIME only
        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")

✅ 1/209 | Saved → 01JAN2023_0545.png
[After 3RIMG_01JAN2023_0545_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.33 GB)
✅ 2/209 | Saved → 01JAN2023_0845.png
[After 3RIMG_01JAN2023_0845_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 3/209 | Saved → 01JAN2023_0715.png
[After 3RIMG_01JAN2023_0715_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 4/209 | Saved → 01JAN2023_0645.png
[After 3RIMG_01JAN2023_0645_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 5/209 | Saved → 01JAN2023_0615.png
[After 3RIMG_01JAN2023_0615_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 6/209 | Saved → 01JAN2023_0815.png
[After 3RIMG_01JAN2023_0815_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 7/209 | Saved → 01JAN2023_0745.png
[After 3RIMG_01JAN2023_0745_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 8/209 | Saved → 02JAN2023_0715.png
[After 3RIMG_02JAN2023_0715_L2G_AOD_V02R00.h5] Memory Used: 41.7% (5.34 GB)
✅ 9/209 | Saved → 02JAN2023_0815.png
[After 3RIMG_02JAN2023_0815_L2G_AOD_V02R00.h5] Memory Used:

In [ ]:
del files

#23/02

In [ ]:
import gc

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 02_feb"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 02"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")

✅ 1/173 | Saved → 01FEB2023_0615.png
✅ 2/173 | Saved → 01FEB2023_0845.png
✅ 3/173 | Saved → 01FEB2023_0545.png
✅ 4/173 | Saved → 01FEB2023_0745.png
✅ 5/173 | Saved → 01FEB2023_0645.png
✅ 6/173 | Saved → 01FEB2023_0815.png
✅ 7/173 | Saved → 02FEB2023_0545.png
✅ 8/173 | Saved → 02FEB2023_0615.png
✅ 9/173 | Saved → 02FEB2023_0745.png
✅ 10/173 | Saved → 02FEB2023_0845.png
✅ 11/173 | Saved → 02FEB2023_0815.png
✅ 12/173 | Saved → 03FEB2023_0545.png
✅ 13/173 | Saved → 03FEB2023_0815.png
✅ 14/173 | Saved → 03FEB2023_0745.png
✅ 15/173 | Saved → 03FEB2023_0615.png
✅ 16/173 | Saved → 03FEB2023_0845.png
✅ 17/173 | Saved → 03FEB2023_0715.png
✅ 18/173 | Saved → 03FEB2023_0645.png
✅ 19/173 | Saved → 04FEB2023_0745.png
✅ 20/173 | Saved → 04FEB2023_0715.png
✅ 21/173 | Saved → 04FEB2023_0545.png
✅ 22/173 | Saved → 04FEB2023_0845.png
✅ 23/173 | Saved → 04FEB2023_0615.png
✅ 24/173 | Saved → 04FEB2023_0815.png
✅ 25/173 | Saved → 04FEB2023_0645.png
✅ 26/173 | Saved → 05FEB2023_0845.png
✅ 27/173 | Saved → 05

In [ ]:
del files

#23/03

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 03_mar"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 03"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")

del files

✅ 1/193 | Saved → 01MAR2023_0615.png
✅ 2/193 | Saved → 01MAR2023_0645.png
✅ 3/193 | Saved → 01MAR2023_0545.png
✅ 4/193 | Saved → 01MAR2023_0715.png
✅ 5/193 | Saved → 01MAR2023_0815.png
✅ 6/193 | Saved → 01MAR2023_0845.png
✅ 7/193 | Saved → 01MAR2023_0745.png
✅ 8/193 | Saved → 02MAR2023_0845.png
✅ 9/193 | Saved → 02MAR2023_0545.png
✅ 10/193 | Saved → 02MAR2023_0715.png
✅ 11/193 | Saved → 02MAR2023_0615.png
✅ 12/193 | Saved → 02MAR2023_0745.png
✅ 13/193 | Saved → 02MAR2023_0815.png
✅ 14/193 | Saved → 02MAR2023_0645.png
✅ 15/193 | Saved → 03MAR2023_0615.png
✅ 16/193 | Saved → 03MAR2023_0815.png
✅ 17/193 | Saved → 03MAR2023_0645.png
✅ 18/193 | Saved → 03MAR2023_0545.png
✅ 19/193 | Saved → 03MAR2023_0745.png
✅ 20/193 | Saved → 03MAR2023_0715.png
✅ 21/193 | Saved → 03MAR2023_0845.png
✅ 22/193 | Saved → 04MAR2023_0815.png
✅ 23/193 | Saved → 04MAR2023_0745.png
✅ 24/193 | Saved → 04MAR2023_0645.png
✅ 25/193 | Saved → 04MAR2023_0715.png
✅ 26/193 | Saved → 04MAR2023_0845.png
✅ 27/193 | Saved → 04

#23/04

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 04_apr"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 04"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ 1/210 | Saved → 01APR2023_0615.png
✅ 2/210 | Saved → 01APR2023_0745.png
✅ 3/210 | Saved → 01APR2023_0845.png
✅ 4/210 | Saved → 01APR2023_0645.png
✅ 5/210 | Saved → 01APR2023_0545.png
✅ 6/210 | Saved → 01APR2023_0815.png
✅ 7/210 | Saved → 01APR2023_0715.png
✅ 8/210 | Saved → 02APR2023_0645.png
✅ 9/210 | Saved → 02APR2023_0715.png
✅ 10/210 | Saved → 02APR2023_0545.png
✅ 11/210 | Saved → 02APR2023_0845.png
✅ 12/210 | Saved → 02APR2023_0745.png
✅ 13/210 | Saved → 02APR2023_0615.png
✅ 14/210 | Saved → 02APR2023_0815.png
✅ 15/210 | Saved → 03APR2023_0545.png
✅ 16/210 | Saved → 03APR2023_0745.png
✅ 17/210 | Saved → 03APR2023_0645.png
✅ 18/210 | Saved → 03APR2023_0815.png
✅ 19/210 | Saved → 03APR2023_0715.png
✅ 20/210 | Saved → 03APR2023_0845.png
✅ 21/210 | Saved → 03APR2023_0615.png
✅ 22/210 | Saved → 04APR2023_0745.png
✅ 23/210 | Saved → 04APR2023_0715.png
✅ 24/210 | Saved → 04APR2023_0545.png
✅ 25/210 | Saved → 04APR2023_0845.png
✅ 26/210 | Saved → 04APR2023_0645.png
✅ 27/210 | Saved → 04

#23/05

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 05_may"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 05"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/184 | Saved → 01MAY2023_0845.png
✅ 2/184 | Saved → 01MAY2023_0545.png
✅ 3/184 | Saved → 01MAY2023_0815.png
✅ 4/184 | Saved → 01MAY2023_0645.png
✅ 5/184 | Saved → 01MAY2023_0715.png
✅ 6/184 | Saved → 01MAY2023_0745.png
✅ 7/184 | Saved → 01MAY2023_0615.png
✅ 8/184 | Saved → 02MAY2023_0615.png
✅ 9/184 | Saved → 02MAY2023_0645.png
✅ 10/184 | Saved → 02MAY2023_0545.png
✅ 11/184 | Saved → 02MAY2023_0815.png
✅ 12/184 | Saved → 02MAY2023_0745.png
✅ 13/184 | Saved → 02MAY2023_0845.png
✅ 14/184 | Saved → 02MAY2023_0715.png
✅ 15/184 | Saved → 03MAY2023_0715.png
✅ 16/184 | Saved → 03MAY2023_0645.png
✅ 17/184 | Saved → 03MAY2023_0745.png
✅ 18/184 | Saved → 03MAY2023_0615.png
✅ 19/184 | Saved → 03MAY2023_0545.png
✅ 20/184 | Saved → 03MAY2023_0815.png
✅ 21/184 | Saved → 03MAY2023_0845.png
✅ 22/184 | Saved → 04MAY2023_0815.png
✅ 23/184 | Saved → 04MAY2023_0645.png
✅ 24/184 | Saved → 04MAY2023_0615.png
✅ 25/184 | Saved → 04MAY2023_0715.png
✅ 26/184 | Saved → 04MAY2023_0845.png
✅ 27/184 | Saved → 04

#23/06

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 06_jun"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 06"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/144 | Saved → 30JUN2025_0545.png
✅ 2/144 | Saved → 30JUN2025_0745.png
✅ 3/144 | Saved → 01JUN2023_0845.png
✅ 4/144 | Saved → 01JUN2023_0815.png
✅ 5/144 | Saved → 01JUN2023_0715.png
✅ 6/144 | Saved → 01JUN2023_0745.png
✅ 7/144 | Saved → 01JUN2023_0645.png
✅ 8/144 | Saved → 01JUN2023_0615.png
✅ 9/144 | Saved → 01JUN2023_0545.png
✅ 10/144 | Saved → 02JUN2023_0815.png
✅ 11/144 | Saved → 02JUN2023_0715.png
✅ 12/144 | Saved → 02JUN2023_0545.png
✅ 13/144 | Saved → 02JUN2023_0645.png
✅ 14/144 | Saved → 02JUN2023_0615.png
✅ 15/144 | Saved → 02JUN2023_0745.png
✅ 16/144 | Saved → 02JUN2023_0845.png
✅ 17/144 | Saved → 03JUN2023_0815.png
✅ 18/144 | Saved → 03JUN2023_0615.png
✅ 19/144 | Saved → 03JUN2023_0745.png
✅ 20/144 | Saved → 03JUN2023_0845.png
✅ 21/144 | Saved → 03JUN2023_0715.png
✅ 22/144 | Saved → 03JUN2023_0645.png
✅ 23/144 | Saved → 04JUN2023_0715.png
✅ 24/144 | Saved → 04JUN2023_0615.png
✅ 25/144 | Saved → 04JUN2023_0545.png
✅ 26/144 | Saved → 04JUN2023_0645.png
✅ 27/144 | Saved → 04

#23/07

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 07_jul"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 07"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ 1/201 | Saved → 01JUL2023_0715.png
✅ 2/201 | Saved → 01JUL2023_0745.png
✅ 3/201 | Saved → 01JUL2023_0545.png
✅ 4/201 | Saved → 01JUL2023_0615.png
✅ 5/201 | Saved → 01JUL2023_0845.png
✅ 6/201 | Saved → 01JUL2023_0815.png
✅ 7/201 | Saved → 01JUL2023_0645.png
✅ 8/201 | Saved → 02JUL2023_0845.png
✅ 9/201 | Saved → 02JUL2023_0815.png
✅ 10/201 | Saved → 02JUL2023_0745.png
✅ 11/201 | Saved → 02JUL2023_0545.png
✅ 12/201 | Saved → 02JUL2023_0715.png
✅ 13/201 | Saved → 02JUL2023_0645.png
✅ 14/201 | Saved → 02JUL2023_0615.png
✅ 15/201 | Saved → 03JUL2023_0715.png
✅ 16/201 | Saved → 03JUL2023_0645.png
✅ 17/201 | Saved → 03JUL2023_0615.png
✅ 18/201 | Saved → 03JUL2023_0545.png
✅ 19/201 | Saved → 03JUL2023_0745.png
✅ 20/201 | Saved → 03JUL2023_0845.png
✅ 21/201 | Saved → 03JUL2023_0815.png
✅ 22/201 | Saved → 04JUL2023_0845.png
✅ 23/201 | Saved → 04JUL2023_0815.png
✅ 24/201 | Saved → 04JUL2023_0545.png
✅ 25/201 | Saved → 04JUL2023_0645.png
✅ 26/201 | Saved → 04JUL2023_0615.png
✅ 27/201 | Saved → 04

#23/08

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 08_aug"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 08"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/211 | Saved → 01AUG2023_0815.png
✅ 2/211 | Saved → 01AUG2023_0645.png
✅ 3/211 | Saved → 01AUG2023_0545.png
✅ 4/211 | Saved → 01AUG2023_0845.png
✅ 5/211 | Saved → 01AUG2023_0745.png
✅ 6/211 | Saved → 01AUG2023_0615.png
✅ 7/211 | Saved → 02AUG2023_0545.png
✅ 8/211 | Saved → 02AUG2023_0745.png
✅ 9/211 | Saved → 02AUG2023_0615.png
✅ 10/211 | Saved → 02AUG2023_0645.png
✅ 11/211 | Saved → 02AUG2023_0845.png
✅ 12/211 | Saved → 03AUG2023_0745.png
✅ 13/211 | Saved → 03AUG2023_0645.png
✅ 14/211 | Saved → 03AUG2023_0815.png
✅ 15/211 | Saved → 03AUG2023_0845.png
✅ 16/211 | Saved → 03AUG2023_0715.png
✅ 17/211 | Saved → 03AUG2023_0615.png
✅ 18/211 | Saved → 03AUG2023_0545.png
✅ 19/211 | Saved → 04AUG2023_0745.png
✅ 20/211 | Saved → 04AUG2023_0645.png
✅ 21/211 | Saved → 04AUG2023_0615.png
✅ 22/211 | Saved → 04AUG2023_0715.png
✅ 23/211 | Saved → 04AUG2023_0545.png
✅ 24/211 | Saved → 04AUG2023_0815.png
✅ 25/211 | Saved → 04AUG2023_0845.png
✅ 26/211 | Saved → 05AUG2023_0645.png
✅ 27/211 | Saved → 05

#23/09

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 09_sept"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 09"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/201 | Saved → 01SEP2023_0545.png
✅ 2/201 | Saved → 01SEP2023_0815.png
✅ 3/201 | Saved → 01SEP2023_0845.png
✅ 4/201 | Saved → 01SEP2023_0745.png
✅ 5/201 | Saved → 01SEP2023_0615.png
✅ 6/201 | Saved → 01SEP2023_0715.png
✅ 7/201 | Saved → 01SEP2023_0645.png
✅ 8/201 | Saved → 02SEP2023_0545.png
✅ 9/201 | Saved → 02SEP2023_0845.png
✅ 10/201 | Saved → 02SEP2023_0645.png
✅ 11/201 | Saved → 02SEP2023_0715.png
✅ 12/201 | Saved → 02SEP2023_0615.png
✅ 13/201 | Saved → 03SEP2023_0745.png
✅ 14/201 | Saved → 03SEP2023_0645.png
✅ 15/201 | Saved → 03SEP2023_0715.png
✅ 16/201 | Saved → 03SEP2023_0845.png
✅ 17/201 | Saved → 03SEP2023_0815.png
✅ 18/201 | Saved → 04SEP2023_0815.png
✅ 19/201 | Saved → 04SEP2023_0715.png
✅ 20/201 | Saved → 04SEP2023_0645.png
✅ 21/201 | Saved → 04SEP2023_0615.png
✅ 22/201 | Saved → 04SEP2023_0845.png
✅ 23/201 | Saved → 04SEP2023_0545.png
✅ 24/201 | Saved → 04SEP2023_0745.png
✅ 25/201 | Saved → 05SEP2023_0715.png
✅ 26/201 | Saved → 05SEP2023_0615.png
✅ 27/201 | Saved → 05

#23/10

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 10_oct"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 10"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/182 | Saved → 01OCT2023_0545.png
✅ 2/182 | Saved → 01OCT2023_0745.png
✅ 3/182 | Saved → 01OCT2023_0715.png
✅ 4/182 | Saved → 01OCT2023_0645.png
✅ 5/182 | Saved → 01OCT2023_0815.png
✅ 6/182 | Saved → 01OCT2023_0615.png
✅ 7/182 | Saved → 01OCT2023_0845.png
✅ 8/182 | Saved → 02OCT2023_0645.png
✅ 9/182 | Saved → 02OCT2023_0545.png
✅ 10/182 | Saved → 02OCT2023_0745.png
✅ 11/182 | Saved → 02OCT2023_0615.png
✅ 12/182 | Saved → 02OCT2023_0715.png
✅ 13/182 | Saved → 02OCT2023_0815.png
✅ 14/182 | Saved → 02OCT2023_0845.png
✅ 15/182 | Saved → 03OCT2023_0645.png
✅ 16/182 | Saved → 03OCT2023_0745.png
✅ 17/182 | Saved → 03OCT2023_0845.png
✅ 18/182 | Saved → 03OCT2023_0545.png
✅ 19/182 | Saved → 03OCT2023_0715.png
✅ 20/182 | Saved → 03OCT2023_0815.png
✅ 21/182 | Saved → 03OCT2023_0615.png
✅ 22/182 | Saved → 04OCT2023_0615.png
✅ 23/182 | Saved → 04OCT2023_0545.png
✅ 24/182 | Saved → 04OCT2023_0815.png
✅ 25/182 | Saved → 04OCT2023_0715.png
✅ 26/182 | Saved → 04OCT2023_0745.png
✅ 27/182 | Saved → 04

#23/11

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 11_nov"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 11"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/203 | Saved → 01NOV2023_0845.png
✅ 2/203 | Saved → 01NOV2023_0815.png
✅ 3/203 | Saved → 01NOV2023_0645.png
✅ 4/203 | Saved → 01NOV2023_0545.png
✅ 5/203 | Saved → 01NOV2023_0615.png
✅ 6/203 | Saved → 01NOV2023_0715.png
✅ 7/203 | Saved → 01NOV2023_0745.png
✅ 8/203 | Saved → 02NOV2023_0545.png
✅ 9/203 | Saved → 02NOV2023_0845.png
✅ 10/203 | Saved → 02NOV2023_0715.png
✅ 11/203 | Saved → 02NOV2023_0615.png
✅ 12/203 | Saved → 02NOV2023_0645.png
✅ 13/203 | Saved → 02NOV2023_0815.png
✅ 14/203 | Saved → 02NOV2023_0745.png
✅ 15/203 | Saved → 03NOV2023_0545.png
✅ 16/203 | Saved → 03NOV2023_0715.png
✅ 17/203 | Saved → 03NOV2023_0615.png
✅ 18/203 | Saved → 03NOV2023_0745.png
✅ 19/203 | Saved → 03NOV2023_0815.png
✅ 20/203 | Saved → 03NOV2023_0645.png
✅ 21/203 | Saved → 03NOV2023_0845.png
✅ 22/203 | Saved → 04NOV2023_0545.png
✅ 23/203 | Saved → 04NOV2023_0615.png
✅ 24/203 | Saved → 04NOV2023_0645.png
✅ 25/203 | Saved → 04NOV2023_0715.png
✅ 26/203 | Saved → 04NOV2023_0745.png
✅ 27/203 | Saved → 04

#23/12

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/23 12_dec"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/23 12"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/183 | Saved → 01DEC2023_0615.png
✅ 2/183 | Saved → 01DEC2023_0845.png
✅ 3/183 | Saved → 01DEC2023_0815.png
✅ 4/183 | Saved → 01DEC2023_0715.png
✅ 5/183 | Saved → 01DEC2023_0745.png
✅ 6/183 | Saved → 01DEC2023_0645.png
✅ 7/183 | Saved → 01DEC2023_0545.png
✅ 8/183 | Saved → 02DEC2023_0645.png
✅ 9/183 | Saved → 02DEC2023_0815.png
✅ 10/183 | Saved → 02DEC2023_0745.png
✅ 11/183 | Saved → 02DEC2023_0545.png
✅ 12/183 | Saved → 02DEC2023_0615.png
✅ 13/183 | Saved → 02DEC2023_0715.png
✅ 14/183 | Saved → 02DEC2023_0845.png
✅ 15/183 | Saved → 06DEC2023_0615.png
✅ 16/183 | Saved → 06DEC2023_0545.png
✅ 17/183 | Saved → 06DEC2023_0715.png
✅ 18/183 | Saved → 06DEC2023_0745.png
✅ 19/183 | Saved → 06DEC2023_0815.png
✅ 20/183 | Saved → 06DEC2023_0845.png
✅ 21/183 | Saved → 06DEC2023_0645.png
✅ 22/183 | Saved → 07DEC2023_0545.png
✅ 23/183 | Saved → 07DEC2023_0815.png
✅ 24/183 | Saved → 07DEC2023_0615.png
✅ 25/183 | Saved → 07DEC2023_0845.png
✅ 26/183 | Saved → 07DEC2023_0645.png
✅ 27/183 | Saved → 07

#24/01

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 01_jan"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 01"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/203 | Saved → 01JAN2024_0645.png
✅ 2/203 | Saved → 01JAN2024_0615.png
✅ 3/203 | Saved → 01JAN2024_0815.png
✅ 4/203 | Saved → 01JAN2024_0715.png
✅ 5/203 | Saved → 01JAN2024_0745.png
✅ 6/203 | Saved → 01JAN2024_0845.png
✅ 7/203 | Saved → 01JAN2024_0545.png
✅ 8/203 | Saved → 02JAN2024_0845.png
✅ 9/203 | Saved → 02JAN2024_0615.png
✅ 10/203 | Saved → 02JAN2024_0815.png
✅ 11/203 | Saved → 02JAN2024_0745.png
✅ 12/203 | Saved → 02JAN2024_0645.png
✅ 13/203 | Saved → 02JAN2024_0715.png
✅ 14/203 | Saved → 02JAN2024_0545.png
✅ 15/203 | Saved → 03JAN2024_0545.png
✅ 16/203 | Saved → 03JAN2024_0615.png
✅ 17/203 | Saved → 03JAN2024_0715.png
✅ 18/203 | Saved → 03JAN2024_0645.png
✅ 19/203 | Saved → 03JAN2024_0815.png
✅ 20/203 | Saved → 03JAN2024_0745.png
✅ 21/203 | Saved → 03JAN2024_0845.png
✅ 22/203 | Saved → 04JAN2024_0645.png
✅ 23/203 | Saved → 04JAN2024_0745.png
✅ 24/203 | Saved → 04JAN2024_0845.png
✅ 25/203 | Saved → 04JAN2024_0615.png
✅ 26/203 | Saved → 04JAN2024_0545.png
✅ 27/203 | Saved → 04

#24/02

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 02_feb"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 02"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ 1/187 | Saved → 01FEB2024_0545.png
✅ 2/187 | Saved → 01FEB2024_0615.png
✅ 3/187 | Saved → 01FEB2024_0845.png
✅ 4/187 | Saved → 01FEB2024_0745.png
✅ 5/187 | Saved → 01FEB2024_0645.png
✅ 6/187 | Saved → 01FEB2024_0815.png
✅ 7/187 | Saved → 01FEB2024_0715.png
✅ 8/187 | Saved → 02FEB2024_0615.png
✅ 9/187 | Saved → 02FEB2024_0545.png
✅ 10/187 | Saved → 02FEB2024_0815.png
✅ 11/187 | Saved → 02FEB2024_0645.png
✅ 12/187 | Saved → 02FEB2024_0715.png
✅ 13/187 | Saved → 02FEB2024_0845.png
✅ 14/187 | Saved → 02FEB2024_0745.png
✅ 15/187 | Saved → 03FEB2024_0615.png
✅ 16/187 | Saved → 03FEB2024_0545.png
✅ 17/187 | Saved → 03FEB2024_0715.png
✅ 18/187 | Saved → 03FEB2024_0815.png
✅ 19/187 | Saved → 03FEB2024_0845.png
✅ 20/187 | Saved → 03FEB2024_0745.png
✅ 21/187 | Saved → 03FEB2024_0645.png
✅ 22/187 | Saved → 04FEB2024_0615.png
✅ 23/187 | Saved → 04FEB2024_0545.png
✅ 24/187 | Saved → 04FEB2024_0715.png
✅ 25/187 | Saved → 04FEB2024_0845.png
✅ 26/187 | Saved → 04FEB2024_0645.png
✅ 27/187 | Saved → 04

#24/03

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 03_mar"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 03"

os.makedirs(output_folder, exist_ok=True)

files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

length = len(files)
counter = 0

for idx, file in enumerate(files, start=1):
    input_path = os.path.join(input_folder, file)

    with h5py.File(input_path, 'r') as f:
        aod = f['AOD'][:].astype(np.float32).squeeze()
        lat = f['latitude'][:].astype(np.float32)
        lon = f['longitude'][:].astype(np.float32)

        lon2d, lat2d = np.meshgrid(lon, lat)

        aod_mumbai = np.where(
            (lat2d >= lat_min) & (lat2d <= lat_max) &
            (lon2d >= lon_min) & (lon2d <= lon_max),
            aod, np.nan
        ).astype(np.float32)

        min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
        aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

        plt.figure(figsize=(8, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

        ax.add_feature(cfeature.LAND)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.RIVERS, alpha=0.5)

        sc = ax.pcolormesh(
            lon2d, lat2d, aod_norm,
            cmap='gray', shading='auto',
            vmin=0.0, vmax=1.0
        )

        cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
        cb.set_label("Normalized AOD (0–1)")

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

        ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
        ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

        out_name = extract_datetime_for_output(file)
        output_path = os.path.join(output_folder, f"{out_name}.png")

        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close("all")

        del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
        gc.collect()

        counter += 1
        print(f"✅ {counter}/{length} | Saved → {out_name}.png")
        #print_memory_usage(tag=f"After {file}")

print("🎉 All files processed and stored in output folder!")
del files

✅ 1/204 | Saved → 01MAR2024_0715.png
✅ 2/204 | Saved → 01MAR2024_0815.png
✅ 3/204 | Saved → 01MAR2024_0645.png
✅ 4/204 | Saved → 01MAR2024_0745.png
✅ 5/204 | Saved → 01MAR2024_0545.png
✅ 6/204 | Saved → 01MAR2024_0615.png
✅ 7/204 | Saved → 02MAR2024_0545.png
✅ 8/204 | Saved → 02MAR2024_0745.png
✅ 9/204 | Saved → 02MAR2024_0815.png
✅ 10/204 | Saved → 02MAR2024_0845.png
✅ 11/204 | Saved → 02MAR2024_0615.png
✅ 12/204 | Saved → 02MAR2024_0715.png
✅ 13/204 | Saved → 02MAR2024_0645.png
✅ 14/204 | Saved → 03MAR2024_0815.png
✅ 15/204 | Saved → 03MAR2024_0545.png
✅ 16/204 | Saved → 03MAR2024_0845.png
✅ 17/204 | Saved → 03MAR2024_0645.png
✅ 18/204 | Saved → 03MAR2024_0715.png
✅ 19/204 | Saved → 03MAR2024_0745.png
✅ 20/204 | Saved → 03MAR2024_0615.png
✅ 21/204 | Saved → 04MAR2024_0545.png
✅ 22/204 | Saved → 04MAR2024_0645.png
✅ 23/204 | Saved → 04MAR2024_0745.png
✅ 24/204 | Saved → 04MAR2024_0815.png
✅ 25/204 | Saved → 04MAR2024_0615.png
✅ 26/204 | Saved → 04MAR2024_0845.png
✅ 27/204 | Saved → 04

#24/04 -- fn here

In [ ]:
def img_conversion(input_folder, output_folder):

    os.makedirs(output_folder, exist_ok=True)

    files = [f for f in os.listdir(input_folder) if f.endswith(".h5")]

    length = len(files)
    counter = 0

    for idx, file in enumerate(files, start=1):
        input_path = os.path.join(input_folder, file)

        with h5py.File(input_path, 'r') as f:
            aod = f['AOD'][:].astype(np.float32).squeeze()
            lat = f['latitude'][:].astype(np.float32)
            lon = f['longitude'][:].astype(np.float32)

            lon2d, lat2d = np.meshgrid(lon, lat)

            aod_mumbai = np.where(
                (lat2d >= lat_min) & (lat2d <= lat_max) &
                (lon2d >= lon_min) & (lon2d <= lon_max),
                aod, np.nan
            ).astype(np.float32)

            min_val, max_val = np.nanmin(aod_mumbai), np.nanmax(aod_mumbai)
            aod_norm = (aod_mumbai - min_val) / (max_val - min_val + 1e-8)

            plt.figure(figsize=(8, 8))
            ax = plt.axes(projection=ccrs.PlateCarree())
            ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

            ax.add_feature(cfeature.LAND)
            ax.add_feature(cfeature.COASTLINE)
            ax.add_feature(cfeature.BORDERS, linestyle=':')
            ax.add_feature(cfeature.RIVERS, alpha=0.5)

            sc = ax.pcolormesh(
                lon2d, lat2d, aod_norm,
                cmap='gray', shading='auto',
                vmin=0.0, vmax=1.0
            )

            cb = plt.colorbar(sc, orientation="vertical", pad=0.02)
            cb.set_label("Normalized AOD (0–1)")

            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")

            ax.set_xticks(np.linspace(lon_min, lon_max, 5), crs=ccrs.PlateCarree())
            ax.set_yticks(np.linspace(lat_min, lat_max, 5), crs=ccrs.PlateCarree())

            out_name = extract_datetime_for_output(file)
            output_path = os.path.join(output_folder, f"{out_name}.png")

            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.close("all")

            del aod, lat, lon, lon2d, lat2d, aod_mumbai, aod_norm, sc, cb
            gc.collect()

            counter += 1
            print(f"✅ {counter}/{length} | Saved → {out_name}.png")
            #print_memory_usage(tag=f"After {file}")

    print("🎉 All files processed and stored in output folder!")
    del files

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 04_apr"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 04"


img_conversion(input_folder, output_folder)

✅ 1/209 | Saved → 01APR2024_0845.png
✅ 2/209 | Saved → 01APR2024_0745.png
✅ 3/209 | Saved → 01APR2024_0615.png
✅ 4/209 | Saved → 01APR2024_0815.png
✅ 5/209 | Saved → 01APR2024_0715.png
✅ 6/209 | Saved → 01APR2024_0545.png
✅ 7/209 | Saved → 01APR2024_0645.png
✅ 8/209 | Saved → 02APR2024_0645.png
✅ 9/209 | Saved → 02APR2024_0745.png
✅ 10/209 | Saved → 02APR2024_0615.png
✅ 11/209 | Saved → 02APR2024_0845.png
✅ 12/209 | Saved → 02APR2024_0815.png
✅ 13/209 | Saved → 02APR2024_0715.png
✅ 14/209 | Saved → 02APR2024_0545.png
✅ 15/209 | Saved → 03APR2024_0545.png
✅ 16/209 | Saved → 03APR2024_0615.png
✅ 17/209 | Saved → 03APR2024_0845.png
✅ 18/209 | Saved → 03APR2024_0745.png
✅ 19/209 | Saved → 03APR2024_0715.png
✅ 20/209 | Saved → 03APR2024_0815.png
✅ 21/209 | Saved → 03APR2024_0645.png
✅ 22/209 | Saved → 04APR2024_0645.png
✅ 23/209 | Saved → 04APR2024_0815.png
✅ 24/209 | Saved → 04APR2024_0845.png
✅ 25/209 | Saved → 04APR2024_0615.png
✅ 26/209 | Saved → 04APR2024_0745.png
✅ 27/209 | Saved → 04

#24/05

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 05_may"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 05"

img_conversion(input_folder, output_folder)

✅ 1/203 | Saved → 01MAY2024_0715.png
✅ 2/203 | Saved → 01MAY2024_0545.png
✅ 3/203 | Saved → 01MAY2024_0645.png
✅ 4/203 | Saved → 01MAY2024_0615.png
✅ 5/203 | Saved → 01MAY2024_0815.png
✅ 6/203 | Saved → 01MAY2024_0745.png
✅ 7/203 | Saved → 01MAY2024_0845.png
✅ 8/203 | Saved → 02MAY2024_0545.png
✅ 9/203 | Saved → 02MAY2024_0615.png
✅ 10/203 | Saved → 02MAY2024_0645.png
✅ 11/203 | Saved → 02MAY2024_0745.png
✅ 12/203 | Saved → 02MAY2024_0845.png
✅ 13/203 | Saved → 02MAY2024_0815.png
✅ 14/203 | Saved → 02MAY2024_0715.png
✅ 15/203 | Saved → 03MAY2024_0545.png
✅ 16/203 | Saved → 03MAY2024_0645.png
✅ 17/203 | Saved → 03MAY2024_0745.png
✅ 18/203 | Saved → 03MAY2024_0845.png
✅ 19/203 | Saved → 03MAY2024_0615.png
✅ 20/203 | Saved → 03MAY2024_0815.png
✅ 21/203 | Saved → 03MAY2024_0715.png
✅ 22/203 | Saved → 04MAY2024_0745.png
✅ 23/203 | Saved → 04MAY2024_0715.png
✅ 24/203 | Saved → 04MAY2024_0815.png
✅ 25/203 | Saved → 04MAY2024_0545.png
✅ 26/203 | Saved → 04MAY2024_0645.png
✅ 27/203 | Saved → 04

#24/06

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 06_jun"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 06"

img_conversion(input_folder, output_folder)

✅ 1/199 | Saved → 01JUN2024_0545.png
✅ 2/199 | Saved → 01JUN2024_0645.png
✅ 3/199 | Saved → 01JUN2024_0745.png
✅ 4/199 | Saved → 01JUN2024_0715.png
✅ 5/199 | Saved → 01JUN2024_0615.png
✅ 6/199 | Saved → 01JUN2024_0815.png
✅ 7/199 | Saved → 01JUN2024_0845.png
✅ 8/199 | Saved → 02JUN2024_0815.png
✅ 9/199 | Saved → 02JUN2024_0545.png
✅ 10/199 | Saved → 02JUN2024_0645.png
✅ 11/199 | Saved → 02JUN2024_0715.png
✅ 12/199 | Saved → 02JUN2024_0615.png
✅ 13/199 | Saved → 02JUN2024_0745.png
✅ 14/199 | Saved → 02JUN2024_0845.png
✅ 15/199 | Saved → 03JUN2024_0815.png
✅ 16/199 | Saved → 03JUN2024_0615.png
✅ 17/199 | Saved → 03JUN2024_0715.png
✅ 18/199 | Saved → 03JUN2024_0745.png
✅ 19/199 | Saved → 03JUN2024_0645.png
✅ 20/199 | Saved → 03JUN2024_0845.png
✅ 21/199 | Saved → 03JUN2024_0545.png
✅ 22/199 | Saved → 04JUN2024_0845.png
✅ 23/199 | Saved → 04JUN2024_0745.png
✅ 24/199 | Saved → 04JUN2024_0545.png
✅ 25/199 | Saved → 04JUN2024_0715.png
✅ 26/199 | Saved → 04JUN2024_0645.png
✅ 27/199 | Saved → 04

#24/07

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 07_jul"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 07"

img_conversion(input_folder, output_folder)

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ 1/205 | Saved → 01JUL2024_0645.png
✅ 2/205 | Saved → 01JUL2024_0615.png
✅ 3/205 | Saved → 01JUL2024_0545.png
✅ 4/205 | Saved → 01JUL2024_0745.png
✅ 5/205 | Saved → 01JUL2024_0815.png
✅ 6/205 | Saved → 01JUL2024_0845.png
✅ 7/205 | Saved → 01JUL2024_0715.png
✅ 8/205 | Saved → 02JUL2024_0745.png
✅ 9/205 | Saved → 02JUL2024_0845.png
✅ 10/205 | Saved → 02JUL2024_0545.png
✅ 11/205 | Saved → 02JUL2024_0815.png
✅ 12/205 | Saved → 02JUL2024_0715.png
✅ 13/205 | Saved → 03JUL2024_0645.png
✅ 14/205 | Saved → 03JUL2024_0815.png
✅ 15/205 | Saved → 03JUL2024_0745.png
✅ 16/205 | Saved → 03JUL2024_0545.png
✅ 17/205 | Saved → 03JUL2024_0615.png
✅ 18/205 | Saved → 03JUL2024_0715.png
✅ 19/205 | Saved → 03JUL2024_0845.png
✅ 20/205 | Saved → 04JUL2024_0545.png
✅ 21/205 | Saved → 04JUL2024_0615.png
✅ 22/205 | Saved → 04JUL2024_0815.png
✅ 23/205 | Saved → 04JUL2024_0845.png
✅ 24/205 | Saved → 04JUL2024_0715.png
✅ 25/205 | Saved → 04JUL2024_0645.png
✅ 26/205 | Saved → 04JUL2024_0745.png
✅ 27/205 | Saved → 05

#24/08

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 08_aug"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 08"

img_conversion(input_folder, output_folder)

✅ 1/189 | Saved → 04AUG2024_0845.png
✅ 2/189 | Saved → 05AUG2024_0615.png
✅ 3/189 | Saved → 05AUG2024_0645.png
✅ 4/189 | Saved → 05AUG2024_0745.png
✅ 5/189 | Saved → 05AUG2024_0715.png
✅ 6/189 | Saved → 05AUG2024_0545.png
✅ 7/189 | Saved → 05AUG2024_0815.png
✅ 8/189 | Saved → 05AUG2024_0845.png
✅ 9/189 | Saved → 06AUG2024_0645.png
✅ 10/189 | Saved → 06AUG2024_0845.png
✅ 11/189 | Saved → 06AUG2024_0745.png
✅ 12/189 | Saved → 06AUG2024_0545.png
✅ 13/189 | Saved → 06AUG2024_0715.png
✅ 14/189 | Saved → 06AUG2024_0615.png
✅ 15/189 | Saved → 06AUG2024_0815.png
✅ 16/189 | Saved → 07AUG2024_0545.png
✅ 17/189 | Saved → 07AUG2024_0615.png
✅ 18/189 | Saved → 07AUG2024_0715.png
✅ 19/189 | Saved → 07AUG2024_0645.png
✅ 20/189 | Saved → 07AUG2024_0815.png
✅ 21/189 | Saved → 07AUG2024_0745.png
✅ 22/189 | Saved → 07AUG2024_0845.png
✅ 23/189 | Saved → 08AUG2024_0745.png
✅ 24/189 | Saved → 08AUG2024_0845.png
✅ 25/189 | Saved → 08AUG2024_0615.png
✅ 26/189 | Saved → 08AUG2024_0715.png
✅ 27/189 | Saved → 08

#24/09

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 09_sept"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 09"

img_conversion(input_folder, output_folder)

✅ 1/207 | Saved → 01SEP2024_0545.png
✅ 2/207 | Saved → 01SEP2024_0615.png
✅ 3/207 | Saved → 01SEP2024_0645.png
✅ 4/207 | Saved → 01SEP2024_0715.png
✅ 5/207 | Saved → 01SEP2024_0815.png
✅ 6/207 | Saved → 01SEP2024_0845.png
✅ 7/207 | Saved → 01SEP2024_0745.png
✅ 8/207 | Saved → 02SEP2024_0545.png
✅ 9/207 | Saved → 02SEP2024_0745.png
✅ 10/207 | Saved → 02SEP2024_0645.png
✅ 11/207 | Saved → 02SEP2024_0845.png
✅ 12/207 | Saved → 02SEP2024_0715.png
✅ 13/207 | Saved → 02SEP2024_0815.png
✅ 14/207 | Saved → 02SEP2024_0615.png
✅ 15/207 | Saved → 03SEP2024_0715.png
✅ 16/207 | Saved → 03SEP2024_0815.png
✅ 17/207 | Saved → 03SEP2024_0545.png
✅ 18/207 | Saved → 03SEP2024_0615.png
✅ 19/207 | Saved → 03SEP2024_0645.png
✅ 20/207 | Saved → 03SEP2024_0745.png
✅ 21/207 | Saved → 03SEP2024_0845.png
✅ 22/207 | Saved → 04SEP2024_0845.png
✅ 23/207 | Saved → 04SEP2024_0745.png
✅ 24/207 | Saved → 04SEP2024_0815.png
✅ 25/207 | Saved → 04SEP2024_0645.png
✅ 26/207 | Saved → 04SEP2024_0545.png
✅ 27/207 | Saved → 04

#24/10

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 10_oct"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 10"

img_conversion(input_folder, output_folder)

✅ 1/212 | Saved → 01OCT2024_0815.png
✅ 2/212 | Saved → 01OCT2024_0545.png
✅ 3/212 | Saved → 01OCT2024_0845.png
✅ 4/212 | Saved → 01OCT2024_0715.png
✅ 5/212 | Saved → 01OCT2024_0645.png
✅ 6/212 | Saved → 01OCT2024_0615.png
✅ 7/212 | Saved → 01OCT2024_0745.png
✅ 8/212 | Saved → 02OCT2024_0545.png
✅ 9/212 | Saved → 02OCT2024_0745.png
✅ 10/212 | Saved → 02OCT2024_0815.png
✅ 11/212 | Saved → 02OCT2024_0845.png
✅ 12/212 | Saved → 02OCT2024_0715.png
✅ 13/212 | Saved → 02OCT2024_0615.png
✅ 14/212 | Saved → 02OCT2024_0645.png
✅ 15/212 | Saved → 03OCT2024_0615.png
✅ 16/212 | Saved → 03OCT2024_0715.png
✅ 17/212 | Saved → 03OCT2024_0845.png
✅ 18/212 | Saved → 03OCT2024_0545.png
✅ 19/212 | Saved → 03OCT2024_0645.png
✅ 20/212 | Saved → 03OCT2024_0745.png
✅ 21/212 | Saved → 03OCT2024_0815.png
✅ 22/212 | Saved → 04OCT2024_0615.png
✅ 23/212 | Saved → 04OCT2024_0545.png
✅ 24/212 | Saved → 04OCT2024_0645.png
✅ 25/212 | Saved → 04OCT2024_0715.png
✅ 26/212 | Saved → 04OCT2024_0745.png
✅ 27/212 | Saved → 04

#24/11

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 11_nov"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 11"

img_conversion(input_folder, output_folder)

✅ 1/198 | Saved → 01NOV2024_0815.png
✅ 2/198 | Saved → 01NOV2024_0645.png
✅ 3/198 | Saved → 01NOV2024_0845.png
✅ 4/198 | Saved → 01NOV2024_0715.png
✅ 5/198 | Saved → 01NOV2024_0545.png
✅ 6/198 | Saved → 01NOV2024_0745.png
✅ 7/198 | Saved → 02NOV2024_0845.png
✅ 8/198 | Saved → 02NOV2024_0715.png
✅ 9/198 | Saved → 02NOV2024_0545.png
✅ 10/198 | Saved → 02NOV2024_0645.png
✅ 11/198 | Saved → 02NOV2024_0745.png
✅ 12/198 | Saved → 02NOV2024_0815.png
✅ 13/198 | Saved → 03NOV2024_0615.png
✅ 14/198 | Saved → 03NOV2024_0645.png
✅ 15/198 | Saved → 03NOV2024_0545.png
✅ 16/198 | Saved → 03NOV2024_0715.png
✅ 17/198 | Saved → 03NOV2024_0745.png
✅ 18/198 | Saved → 03NOV2024_0845.png
✅ 19/198 | Saved → 03NOV2024_0815.png
✅ 20/198 | Saved → 04NOV2024_0715.png
✅ 21/198 | Saved → 04NOV2024_0645.png
✅ 22/198 | Saved → 04NOV2024_0615.png
✅ 23/198 | Saved → 04NOV2024_0545.png
✅ 24/198 | Saved → 04NOV2024_0815.png
✅ 25/198 | Saved → 04NOV2024_0745.png
✅ 26/198 | Saved → 04NOV2024_0845.png
✅ 27/198 | Saved → 05

#24/12

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/24 12_dec"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/24 12"

img_conversion(input_folder, output_folder)

✅ 1/200 | Saved → 01DEC2024_0545.png
✅ 2/200 | Saved → 01DEC2024_0845.png
✅ 3/200 | Saved → 01DEC2024_0715.png
✅ 4/200 | Saved → 01DEC2024_0745.png
✅ 5/200 | Saved → 01DEC2024_0815.png
✅ 6/200 | Saved → 01DEC2024_0615.png
✅ 7/200 | Saved → 01DEC2024_0645.png
✅ 8/200 | Saved → 02DEC2024_0615.png
✅ 9/200 | Saved → 02DEC2024_0545.png
✅ 10/200 | Saved → 02DEC2024_0715.png
✅ 11/200 | Saved → 02DEC2024_0745.png
✅ 12/200 | Saved → 02DEC2024_0815.png
✅ 13/200 | Saved → 02DEC2024_0845.png
✅ 14/200 | Saved → 02DEC2024_0645.png
✅ 15/200 | Saved → 03DEC2024_0615.png
✅ 16/200 | Saved → 03DEC2024_0645.png
✅ 17/200 | Saved → 03DEC2024_0845.png
✅ 18/200 | Saved → 03DEC2024_0715.png
✅ 19/200 | Saved → 04DEC2024_0715.png
✅ 20/200 | Saved → 04DEC2024_0645.png
✅ 21/200 | Saved → 04DEC2024_0745.png
✅ 22/200 | Saved → 04DEC2024_0815.png
✅ 23/200 | Saved → 04DEC2024_0545.png
✅ 24/200 | Saved → 04DEC2024_0615.png
✅ 25/200 | Saved → 04DEC2024_0845.png
✅ 26/200 | Saved → 05DEC2024_0545.png
✅ 27/200 | Saved → 05

#25/01

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 01_jan"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 01"

img_conversion(input_folder, output_folder)

✅ 1/200 | Saved → 01JAN2025_0545.png
✅ 2/200 | Saved → 01JAN2025_0845.png
✅ 3/200 | Saved → 01JAN2025_0645.png
✅ 4/200 | Saved → 01JAN2025_0615.png
✅ 5/200 | Saved → 01JAN2025_0715.png
✅ 6/200 | Saved → 01JAN2025_0815.png
✅ 7/200 | Saved → 01JAN2025_0745.png
✅ 8/200 | Saved → 02JAN2025_0615.png
✅ 9/200 | Saved → 02JAN2025_0645.png
✅ 10/200 | Saved → 02JAN2025_0745.png
✅ 11/200 | Saved → 02JAN2025_0545.png
✅ 12/200 | Saved → 02JAN2025_0715.png
✅ 13/200 | Saved → 02JAN2025_0815.png
✅ 14/200 | Saved → 02JAN2025_0845.png
✅ 15/200 | Saved → 03JAN2025_0545.png
✅ 16/200 | Saved → 03JAN2025_0745.png
✅ 17/200 | Saved → 03JAN2025_0845.png
✅ 18/200 | Saved → 03JAN2025_0615.png
✅ 19/200 | Saved → 03JAN2025_0815.png
✅ 20/200 | Saved → 03JAN2025_0645.png
✅ 21/200 | Saved → 03JAN2025_0715.png
✅ 22/200 | Saved → 04JAN2025_0715.png
✅ 23/200 | Saved → 04JAN2025_0645.png
✅ 24/200 | Saved → 04JAN2025_0615.png
✅ 25/200 | Saved → 04JAN2025_0745.png
✅ 26/200 | Saved → 04JAN2025_0845.png
✅ 27/200 | Saved → 04

#25/02

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 02_feb"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 02"

img_conversion(input_folder, output_folder)

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅ 1/155 | Saved → 01FEB2025_0815.png
✅ 2/155 | Saved → 01FEB2025_0545.png
✅ 3/155 | Saved → 01FEB2025_0615.png
✅ 4/155 | Saved → 01FEB2025_0845.png
✅ 5/155 | Saved → 01FEB2025_0745.png
✅ 6/155 | Saved → 02FEB2025_0715.png
✅ 7/155 | Saved → 02FEB2025_0815.png
✅ 8/155 | Saved → 02FEB2025_0615.png
✅ 9/155 | Saved → 02FEB2025_0745.png
✅ 10/155 | Saved → 02FEB2025_0845.png
✅ 11/155 | Saved → 03FEB2025_0545.png
✅ 12/155 | Saved → 03FEB2025_0715.png
✅ 13/155 | Saved → 03FEB2025_0815.png
✅ 14/155 | Saved → 03FEB2025_0645.png
✅ 15/155 | Saved → 03FEB2025_0745.png
✅ 16/155 | Saved → 03FEB2025_0845.png
✅ 17/155 | Saved → 04FEB2025_0545.png
✅ 18/155 | Saved → 04FEB2025_0845.png
✅ 19/155 | Saved → 04FEB2025_0645.png
✅ 20/155 | Saved → 04FEB2025_0715.png
✅ 21/155 | Saved → 04FEB2025_0615.png
✅ 22/155 | Saved → 04FEB2025_0745.png
✅ 23/155 | Saved → 04FEB2025_0815.png
✅ 24/155 | Saved → 05FEB2025_0545.png
✅ 25/155 | Saved → 05FEB2025_0815.png
✅ 26/155 | Saved → 05FEB2025_0615.png
✅ 27/155 | Saved → 05

#25/03

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 03_mar"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 03"

img_conversion(input_folder, output_folder)

✅ 1/182 | Saved → 01MAR2025_0545.png
✅ 2/182 | Saved → 01MAR2025_0715.png
✅ 3/182 | Saved → 01MAR2025_0845.png
✅ 4/182 | Saved → 01MAR2025_0815.png
✅ 5/182 | Saved → 01MAR2025_0645.png
✅ 6/182 | Saved → 01MAR2025_0745.png
✅ 7/182 | Saved → 02MAR2025_0545.png
✅ 8/182 | Saved → 02MAR2025_0745.png
✅ 9/182 | Saved → 02MAR2025_0815.png
✅ 10/182 | Saved → 02MAR2025_0645.png
✅ 11/182 | Saved → 02MAR2025_0845.png
✅ 12/182 | Saved → 02MAR2025_0715.png
✅ 13/182 | Saved → 03MAR2025_0545.png
✅ 14/182 | Saved → 03MAR2025_0815.png
✅ 15/182 | Saved → 03MAR2025_0845.png
✅ 16/182 | Saved → 03MAR2025_0715.png
✅ 17/182 | Saved → 03MAR2025_0645.png
✅ 18/182 | Saved → 03MAR2025_0745.png
✅ 19/182 | Saved → 04MAR2025_0545.png
✅ 20/182 | Saved → 04MAR2025_0645.png
✅ 21/182 | Saved → 04MAR2025_0715.png
✅ 22/182 | Saved → 04MAR2025_0845.png
✅ 23/182 | Saved → 04MAR2025_0815.png
✅ 24/182 | Saved → 04MAR2025_0745.png
✅ 25/182 | Saved → 05MAR2025_0545.png
✅ 26/182 | Saved → 05MAR2025_0715.png
✅ 27/182 | Saved → 05

#25/04

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 04_apr"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 04"

img_conversion(input_folder, output_folder)

✅ 1/180 | Saved → 01APR2025_0545.png
✅ 2/180 | Saved → 01APR2025_0645.png
✅ 3/180 | Saved → 01APR2025_0715.png
✅ 4/180 | Saved → 01APR2025_0815.png
✅ 5/180 | Saved → 01APR2025_0745.png
✅ 6/180 | Saved → 01APR2025_0845.png
✅ 7/180 | Saved → 02APR2025_0715.png
✅ 8/180 | Saved → 02APR2025_0545.png
✅ 9/180 | Saved → 02APR2025_0645.png
✅ 10/180 | Saved → 02APR2025_0745.png
✅ 11/180 | Saved → 02APR2025_0845.png
✅ 12/180 | Saved → 02APR2025_0815.png
✅ 13/180 | Saved → 03APR2025_0545.png
✅ 14/180 | Saved → 03APR2025_0745.png
✅ 15/180 | Saved → 03APR2025_0845.png
✅ 16/180 | Saved → 03APR2025_0815.png
✅ 17/180 | Saved → 03APR2025_0715.png
✅ 18/180 | Saved → 03APR2025_0645.png
✅ 19/180 | Saved → 04APR2025_0545.png
✅ 20/180 | Saved → 04APR2025_0815.png
✅ 21/180 | Saved → 04APR2025_0745.png
✅ 22/180 | Saved → 04APR2025_0715.png
✅ 23/180 | Saved → 04APR2025_0645.png
✅ 24/180 | Saved → 04APR2025_0845.png
✅ 25/180 | Saved → 05APR2025_0715.png
✅ 26/180 | Saved → 05APR2025_0745.png
✅ 27/180 | Saved → 05

#25/05

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 05_may"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 05"

img_conversion(input_folder, output_folder)

✅ 1/217 | Saved → 01MAY2025_0545.png
✅ 2/217 | Saved → 01MAY2025_0615.png
✅ 3/217 | Saved → 01MAY2025_0845.png
✅ 4/217 | Saved → 01MAY2025_0715.png
✅ 5/217 | Saved → 01MAY2025_0815.png
✅ 6/217 | Saved → 01MAY2025_0645.png
✅ 7/217 | Saved → 01MAY2025_0745.png
✅ 8/217 | Saved → 02MAY2025_0815.png
✅ 9/217 | Saved → 02MAY2025_0545.png
✅ 10/217 | Saved → 02MAY2025_0715.png
✅ 11/217 | Saved → 02MAY2025_0745.png
✅ 12/217 | Saved → 02MAY2025_0645.png
✅ 13/217 | Saved → 02MAY2025_0615.png
✅ 14/217 | Saved → 02MAY2025_0845.png
✅ 15/217 | Saved → 03MAY2025_0545.png
✅ 16/217 | Saved → 03MAY2025_0715.png
✅ 17/217 | Saved → 03MAY2025_0845.png
✅ 18/217 | Saved → 03MAY2025_0745.png
✅ 19/217 | Saved → 03MAY2025_0645.png
✅ 20/217 | Saved → 03MAY2025_0615.png
✅ 21/217 | Saved → 03MAY2025_0815.png
✅ 22/217 | Saved → 04MAY2025_0645.png
✅ 23/217 | Saved → 04MAY2025_0745.png
✅ 24/217 | Saved → 04MAY2025_0615.png
✅ 25/217 | Saved → 04MAY2025_0845.png
✅ 26/217 | Saved → 04MAY2025_0815.png
✅ 27/217 | Saved → 04

#25/06

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 06_jun"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 06"

img_conversion(input_folder, output_folder)

✅ 1/204 | Saved → 01JUN2025_0545.png
✅ 2/204 | Saved → 01JUN2025_0715.png
✅ 3/204 | Saved → 01JUN2025_0645.png
✅ 4/204 | Saved → 01JUN2025_0815.png
✅ 5/204 | Saved → 01JUN2025_0745.png
✅ 6/204 | Saved → 01JUN2025_0845.png
✅ 7/204 | Saved → 01JUN2025_0615.png
✅ 8/204 | Saved → 02JUN2025_0615.png
✅ 9/204 | Saved → 02JUN2025_0815.png
✅ 10/204 | Saved → 02JUN2025_0715.png
✅ 11/204 | Saved → 02JUN2025_0645.png
✅ 12/204 | Saved → 02JUN2025_0745.png
✅ 13/204 | Saved → 02JUN2025_0845.png
✅ 14/204 | Saved → 03JUN2025_0545.png
✅ 15/204 | Saved → 03JUN2025_0615.png
✅ 16/204 | Saved → 03JUN2025_0715.png
✅ 17/204 | Saved → 03JUN2025_0645.png
✅ 18/204 | Saved → 03JUN2025_0845.png
✅ 19/204 | Saved → 04JUN2025_0615.png
✅ 20/204 | Saved → 04JUN2025_0545.png
✅ 21/204 | Saved → 04JUN2025_0645.png
✅ 22/204 | Saved → 04JUN2025_0745.png
✅ 23/204 | Saved → 04JUN2025_0845.png
✅ 24/204 | Saved → 04JUN2025_0715.png
✅ 25/204 | Saved → 04JUN2025_0815.png
✅ 26/204 | Saved → 05JUN2025_0545.png
✅ 27/204 | Saved → 05

#25/07

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 07_jul"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 07"

img_conversion(input_folder, output_folder)

✅ 1/215 | Saved → 01JUL2025_0645.png
✅ 2/215 | Saved → 01JUL2025_0715.png
✅ 3/215 | Saved → 01JUL2025_0845.png
✅ 4/215 | Saved → 01JUL2025_0815.png
✅ 5/215 | Saved → 01JUL2025_0745.png
✅ 6/215 | Saved → 02JUL2025_0615.png
✅ 7/215 | Saved → 02JUL2025_0845.png
✅ 8/215 | Saved → 02JUL2025_0815.png
✅ 9/215 | Saved → 02JUL2025_0715.png
✅ 10/215 | Saved → 02JUL2025_0545.png
✅ 11/215 | Saved → 02JUL2025_0745.png
✅ 12/215 | Saved → 02JUL2025_0645.png
✅ 13/215 | Saved → 03JUL2025_0645.png
✅ 14/215 | Saved → 03JUL2025_0745.png
✅ 15/215 | Saved → 03JUL2025_0845.png
✅ 16/215 | Saved → 03JUL2025_0615.png
✅ 17/215 | Saved → 03JUL2025_0545.png
✅ 18/215 | Saved → 03JUL2025_0715.png
✅ 19/215 | Saved → 03JUL2025_0815.png
✅ 20/215 | Saved → 04JUL2025_0545.png
✅ 21/215 | Saved → 04JUL2025_0715.png
✅ 22/215 | Saved → 04JUL2025_0645.png
✅ 23/215 | Saved → 04JUL2025_0815.png
✅ 24/215 | Saved → 04JUL2025_0845.png
✅ 25/215 | Saved → 04JUL2025_0615.png
✅ 26/215 | Saved → 04JUL2025_0745.png
✅ 27/215 | Saved → 05

#25/08

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 08_aug"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 08"

img_conversion(input_folder, output_folder)

✅ 1/216 | Saved → 01AUG2025_0845.png
✅ 2/216 | Saved → 01AUG2025_0745.png
✅ 3/216 | Saved → 01AUG2025_0545.png
✅ 4/216 | Saved → 01AUG2025_0615.png
✅ 5/216 | Saved → 01AUG2025_0645.png
✅ 6/216 | Saved → 01AUG2025_0815.png
✅ 7/216 | Saved → 01AUG2025_0715.png
✅ 8/216 | Saved → 02AUG2025_0615.png
✅ 9/216 | Saved → 02AUG2025_0645.png
✅ 10/216 | Saved → 02AUG2025_0715.png
✅ 11/216 | Saved → 02AUG2025_0845.png
✅ 12/216 | Saved → 02AUG2025_0745.png
✅ 13/216 | Saved → 02AUG2025_0815.png
✅ 14/216 | Saved → 03AUG2025_0545.png
✅ 15/216 | Saved → 03AUG2025_0615.png
✅ 16/216 | Saved → 03AUG2025_0715.png
✅ 17/216 | Saved → 03AUG2025_0815.png
✅ 18/216 | Saved → 03AUG2025_0645.png
✅ 19/216 | Saved → 03AUG2025_0845.png
✅ 20/216 | Saved → 03AUG2025_0745.png
✅ 21/216 | Saved → 04AUG2025_0545.png
✅ 22/216 | Saved → 04AUG2025_0745.png
✅ 23/216 | Saved → 04AUG2025_0715.png
✅ 24/216 | Saved → 04AUG2025_0845.png
✅ 25/216 | Saved → 04AUG2025_0815.png
✅ 26/216 | Saved → 04AUG2025_0615.png
✅ 27/216 | Saved → 04

#25/09

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 09_sept"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 09"

img_conversion(input_folder, output_folder)

✅ 1/208 | Saved → 01SEP2025_0615.png
✅ 2/208 | Saved → 01SEP2025_0545.png
✅ 3/208 | Saved → 01SEP2025_0715.png
✅ 4/208 | Saved → 01SEP2025_0845.png
✅ 5/208 | Saved → 01SEP2025_0815.png
✅ 6/208 | Saved → 01SEP2025_0745.png
✅ 7/208 | Saved → 01SEP2025_0645.png
✅ 8/208 | Saved → 02SEP2025_0745.png
✅ 9/208 | Saved → 02SEP2025_0845.png
✅ 10/208 | Saved → 02SEP2025_0815.png
✅ 11/208 | Saved → 02SEP2025_0615.png
✅ 12/208 | Saved → 02SEP2025_0545.png
✅ 13/208 | Saved → 02SEP2025_0715.png
✅ 14/208 | Saved → 02SEP2025_0645.png
✅ 15/208 | Saved → 03SEP2025_0545.png
✅ 16/208 | Saved → 03SEP2025_0845.png
✅ 17/208 | Saved → 03SEP2025_0645.png
✅ 18/208 | Saved → 03SEP2025_0815.png
✅ 19/208 | Saved → 03SEP2025_0615.png
✅ 20/208 | Saved → 03SEP2025_0715.png
✅ 21/208 | Saved → 03SEP2025_0745.png
✅ 22/208 | Saved → 04SEP2025_0545.png
✅ 23/208 | Saved → 04SEP2025_0715.png
✅ 24/208 | Saved → 04SEP2025_0815.png
✅ 25/208 | Saved → 04SEP2025_0745.png
✅ 26/208 | Saved → 04SEP2025_0645.png
✅ 27/208 | Saved → 04

#25/10

In [ ]:
input_folder = "/content/drive/MyDrive/Amajor/img: inp- code- op/monthwise_data/25 10_oct"
output_folder = "/content/drive/MyDrive/Amajor/img_summ/op_img_post_conversion/25 10"

img_conversion(input_folder, output_folder)

✅ 1/215 | Saved → 01OCT2025_0645.png
✅ 2/215 | Saved → 01OCT2025_0545.png
✅ 3/215 | Saved → 01OCT2025_0615.png
✅ 4/215 | Saved → 01OCT2025_0745.png
✅ 5/215 | Saved → 01OCT2025_0715.png
✅ 6/215 | Saved → 01OCT2025_0845.png
✅ 7/215 | Saved → 01OCT2025_0815.png
✅ 8/215 | Saved → 02OCT2025_0615.png
✅ 9/215 | Saved → 02OCT2025_0545.png
✅ 10/215 | Saved → 02OCT2025_0845.png
✅ 11/215 | Saved → 02OCT2025_0715.png
✅ 12/215 | Saved → 02OCT2025_0645.png
✅ 13/215 | Saved → 02OCT2025_0815.png
✅ 14/215 | Saved → 02OCT2025_0745.png
✅ 15/215 | Saved → 03OCT2025_0545.png
✅ 16/215 | Saved → 03OCT2025_0645.png
✅ 17/215 | Saved → 03OCT2025_0815.png
✅ 18/215 | Saved → 03OCT2025_0745.png
✅ 19/215 | Saved → 03OCT2025_0615.png
✅ 20/215 | Saved → 03OCT2025_0715.png
✅ 21/215 | Saved → 03OCT2025_0845.png
✅ 22/215 | Saved → 04OCT2025_0545.png
✅ 23/215 | Saved → 04OCT2025_0745.png
✅ 24/215 | Saved → 04OCT2025_0715.png
✅ 25/215 | Saved → 04OCT2025_0815.png
✅ 26/215 | Saved → 04OCT2025_0645.png
✅ 27/215 | Saved → 04

In [ ]:
'hi'

'hi'